In [1]:
%pip install langchain langchain-openai pandas tqdm

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [11]:
import json
import os
import pandas as pd
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import JsonOutputParser
from langchain_core.exceptions import OutputParserException

# ==========================================
# 1. CẤU HÌNH LLM ENDPOINT VỚI LANGCHAIN
# ==========================================
llm = ChatOpenAI(
    base_url="https://llm.phuocnguyn.id.vn/v1",
    api_key="sk-dummy",
    model="google/gemma-3-27b-it-qat-q4_0-gguf:Q4_0",
    temperature=0.1,
    max_tokens=2048,
    request_timeout=60, # Tự động ngắt nếu server treo quá 60 giây
    default_headers={
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36",
        "Accept": "application/json"
    }
)

# ==========================================
# 2. XÂY DỰNG PROMPT VÀ OUTPUT PARSER
# ==========================================
parser = JsonOutputParser()

prompt_template = """Bạn là một chuyên gia giáo dục Toán Tiểu học dày dặn kinh nghiệm. 
Hãy đọc Đề bài và Lời giải dưới đây. Nhiệm vụ của bạn là dự đoán NHỮNG SAI LẦM PHỔ BIẾN NHẤT (tối đa 3 lỗi cho toàn bộ bài toán) mà một học sinh lớp 5 thường mắc phải. KHÔNG ĐƯỢC bịa ra các lỗi phi lý.

Phân loại các lỗi đó vào 4 nhóm sau (chỉ ghi số lượng, tối đa mỗi nhóm không quá 2 lỗi):
1. Arithmetic: Tính nhầm phép tính đơn giản (cộng trừ nhân chia sai).
2. Procedural: Áp dụng đúng công thức nhưng sai thứ tự bước, quên chia 2, quên đổi đơn vị.
3. Conceptual: Hiểu sai hoàn toàn bản chất (VD: Đề yêu cầu Cộng lại đi Nhân, dùng thừa dữ kiện rác, nhầm chu vi thành diện tích).
4. Lack_of_Sense: Lỗi đọc hiểu, dịch sai câu chữ thành phép toán.

Đề bài:
{content}

Lời giải:
{analysis}

CHỈ TRẢ VỀ ĐÚNG MỘT OBJECT JSON THEO ĐỊNH DẠNG SAU:
{{
    "Arithmetic": 0,
    "Procedural": 0,
    "Conceptual": 0,
    "Lack_of_Sense": 0
}}"""

prompt = PromptTemplate(
    template=prompt_template,
    input_variables=["content", "analysis"]
)

chain = prompt | llm | parser

# ==========================================
# 3. CÁC HÀM TIỆN ÍCH
# ==========================================
def calculate_misconception_score(misconception_dict):
    n_arith = float(misconception_dict.get("Arithmetic", 0))
    n_proc = float(misconception_dict.get("Procedural", 0))
    n_conc = float(misconception_dict.get("Conceptual", 0))
    n_lack = float(misconception_dict.get("Lack_of_Sense", 0))
    
    score = (0.2 * n_arith) + (0.5 * n_proc) + (0.8 * n_conc) + (1.0 * n_lack)
    return round(score, 3)

def process_single_item(item_id, item_data):
    """Hàm xử lý độc lập cho 1 câu hỏi để chạy đa luồng"""
    content = item_data.get("content", "")
    analysis = item_data.get("analysis", "")
    
    row_data = {
        "ID": item_id,
        "LLM_Arithmetic": 0,
        "LLM_Procedural": 0,
        "LLM_Conceptual": 0,
        "LLM_Lack_of_Sense": 0,
        "LLM_Misconception_Score": 0.0
    }

    if content and analysis:
        try:
            # Gọi LLM
            response_json = chain.invoke({"content": content, "analysis": analysis})
            
            row_data["LLM_Arithmetic"] = response_json.get("Arithmetic", 0)
            row_data["LLM_Procedural"] = response_json.get("Procedural", 0)
            row_data["LLM_Conceptual"] = response_json.get("Conceptual", 0)
            row_data["LLM_Lack_of_Sense"] = response_json.get("Lack_of_Sense", 0)
            row_data["LLM_Misconception_Score"] = calculate_misconception_score(response_json)
            
        except Exception as e:
            # THAY ĐỔI: In thẳng lỗi ra màn hình để biết tại sao LLM không chạy
            print(f"\n[LỖI API ở ID {item_id}]: {str(e)}") 
            
    return row_data

# ==========================================
# 4. CHẠY PIPELINE VỚI CƠ CHẾ CHECKPOINT
# ==========================================
def extract_llm_with_checkpoints(json_filepath, output_dir, chunk_size=500, max_workers=5):
    """
    Chạy đa luồng, tự động lưu file sau mỗi chunk_size câu hỏi.
    Tự động bỏ qua các file đã lưu trước đó.
    """
    # Tạo thư mục chứa file tạm nếu chưa có
    os.makedirs(output_dir, exist_ok=True)
    
    print(f"Đang đọc dữ liệu từ: {json_filepath}")
    with open(json_filepath, 'r', encoding='utf-8') as f:
        data = json.load(f)
        
    # Chuyển đổi JSON object thành list các tuple (ID, Data) để dễ chia nhỏ
    items_list = list(data.items())
    total_items = len(items_list)
    print(f"Tổng số câu hỏi: {total_items}")

    # Chạy vòng lặp theo từng cụm (chunk)
    for start_idx in range(0, total_items, chunk_size):
        end_idx = min(start_idx + chunk_size, total_items)
        part_number = (start_idx // chunk_size) + 1
        output_filename = os.path.join(output_dir, f"llm_part_{part_number}.csv")
        
        # 1. Cơ chế Resume: Nếu file part này đã tồn tại, bỏ qua luôn cụm này
        if os.path.exists(output_filename):
            print(f"⏩ Đã tìm thấy {output_filename}, tự động bỏ qua câu {start_idx} đến {end_idx-1}.")
            continue
            
        print(f"\n⏳ Đang xử lý CỤM {part_number} (từ câu {start_idx} đến {end_idx-1})...")
        chunk_items = items_list[start_idx:end_idx]
        chunk_results = []
        
        # 2. Xử lý Đa luồng cho cụm hiện tại
        with ThreadPoolExecutor(max_workers=max_workers) as executor:
            futures = {executor.submit(process_single_item, item_id, item_data): item_id 
                       for item_id, item_data in chunk_items}
            
            for future in tqdm(as_completed(futures), total=len(chunk_items), desc=f"Tiến độ Cụm {part_number}"):
                result = future.result()
                chunk_results.append(result)
                
        # 3. Lưu cụm vừa hoàn thành ra file CSV
        df_chunk = pd.DataFrame(chunk_results)
        df_chunk.to_csv(output_filename, index=False, encoding='utf-8-sig')
        print(f"💾 Đã lưu thành công: {output_filename}")

# ==========================================
# 5. HÀM GỘP TẤT CẢ CÁC PART THÀNH 1 FILE CHÍNH
# ==========================================
def merge_all_checkpoints(output_dir, final_csv):
    print(f"\n🔄 Đang tiến hành gộp tất cả các file trong thư mục {output_dir}...")
    all_files = [f for f in os.listdir(output_dir) if f.startswith("llm_part_") and f.endswith(".csv")]
    
    if not all_files:
        print("❌ Không tìm thấy file checkpoint nào để gộp.")
        return
        
    df_list = []
    for file in all_files:
        filepath = os.path.join(output_dir, file)
        df = pd.read_csv(filepath)
        df_list.append(df)
        
    if not df_list:
        print("❌ Các file CSV đều trống rỗng.")
        return

    # 1. Gộp DataFrame (BẮT BUỘC nằm ngoài khối try..except để luôn được khởi tạo)
    df_final = pd.concat(df_list, ignore_index=True)
    
    # 2. Xử lý sắp xếp ID an toàn
    try:
        # Dùng một biến tạm để sort, nếu sập thì df_final gốc vẫn còn nguyên vẹn
        df_temp = df_final.copy()
        df_temp['ID_Num'] = pd.to_numeric(df_temp['ID'], errors='coerce')
        df_final = df_temp.sort_values(by='ID_Num').drop(columns=['ID_Num'])
    except Exception as e:
        print(f"⚠️ Bỏ qua bước sắp xếp ID (Chi tiết: {e})")
        
    # 3. Lưu ra file CSV cuối cùng
    df_final.to_csv(final_csv, index=False, encoding='utf-8-sig')
    print(f"✅ HOÀN TẤT TOÀN BỘ! File gộp cuối cùng lưu tại: {final_csv} (Tổng số dòng: {len(df_final)})")

# ==========================================
# CHẠY CHƯƠNG TRÌNH
# ==========================================
if __name__ == "__main__":
    # ĐƯỜNG DẪN CỦA BẠN (Hãy thay đổi cho đúng)
    INPUT_FILE = "D:/IntelligentTesting/IntelligentTesting/XES3G5M/XES3G5M/metadata/question_full.json"
    CHECKPOINT_DIR = "llm_checkpoints" # Thư mục chứa các file llm_part_1.csv, llm_part_2.csv...
    FINAL_OUTPUT_CSV = "llm_misconceptions_full.csv"
    
    # 1. Chạy trích xuất (Tự động lưu mỗi 500 câu)
    extract_llm_with_checkpoints(INPUT_FILE, CHECKPOINT_DIR, chunk_size=500, max_workers=5)
    
    # 2. Sau khi chạy xong toàn bộ, gọi hàm gộp các file part lại
    merge_all_checkpoints(CHECKPOINT_DIR, FINAL_OUTPUT_CSV)

Đang đọc dữ liệu từ: D:/IntelligentTesting/IntelligentTesting/XES3G5M/XES3G5M/metadata/question_full.json
Tổng số câu hỏi: 7172

⏳ Đang xử lý CỤM 1 (từ câu 0 đến 499)...


Tiến độ Cụm 1: 100%|██████████| 500/500 [21:49<00:00,  2.62s/it]


💾 Đã lưu thành công: llm_checkpoints\llm_part_1.csv

⏳ Đang xử lý CỤM 2 (từ câu 500 đến 999)...


Tiến độ Cụm 2: 100%|██████████| 500/500 [23:58<00:00,  2.88s/it]


💾 Đã lưu thành công: llm_checkpoints\llm_part_2.csv

⏳ Đang xử lý CỤM 3 (từ câu 1000 đến 1499)...


...

💾 Đã lưu thành công: llm_checkpoints\llm_part_13.csv

⏳ Đang xử lý CỤM 14 (từ câu 6500 đến 6999)...


Tiến độ Cụm 14: 100%|██████████| 500/500 [23:08<00:00,  2.78s/it]


💾 Đã lưu thành công: llm_checkpoints\llm_part_14.csv

⏳ Đang xử lý CỤM 15 (từ câu 7000 đến 7171)...


Tiến độ Cụm 15: 100%|██████████| 172/172 [07:03<00:00,  2.46s/it]


💾 Đã lưu thành công: llm_checkpoints\llm_part_15.csv

🔄 Đang tiến hành gộp tất cả các file trong thư mục llm_checkpoints...
✅ HOÀN TẤT TOÀN BỘ! File gộp cuối cùng lưu tại: llm_misconceptions_full.csv (Tổng số dòng: 7172)
